# MMA-TMS: chạy AI worker (YOLO pose) trên Colab

Bản chạy thử. Kế hoạch đầy đủ: `docs/AI-WORKER-COLAB-PLAN.md`.

1. Runtime > Change runtime type > **T4 GPU**.
2. Biểu tượng chìa khoá (Secrets) bên trái: nhập `REDIS_URL`, `NESTJS_API_URL`, `WORKER_SECRET_TOKEN`, `SUPABASE_URL`, `SUPABASE_SERVICE_KEY` (tuỳ chọn `YOLO_MODEL`, `POLL_INTERVAL_S`), bật **Notebook access**.
3. Chạy lần lượt từng ô. Ô cuối chạy worker cho tới khi bấm Stop.

Notebook không in và không ghi secret ra file. Output của worker đi qua bộ lọc che mọi giá trị secret. Đừng chia sẻ notebook kèm output.

In [ ]:
# 1. GPU
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() if gpu.returncode == 0 else "không có, worker sẽ chạy CPU (chậm hơn nhiều)")

In [ ]:
# 2. Secrets (chỉ đọc từ Colab Secrets, không hardcode)
from google.colab import userdata

REQUIRED = ["REDIS_URL", "NESTJS_API_URL", "WORKER_SECRET_TOKEN", "SUPABASE_URL", "SUPABASE_SERVICE_KEY"]
OPTIONAL = {"YOLO_MODEL": "yolov8n-pose", "POLL_INTERVAL_S": "3"}

def read_secret(name):
    try:
        return (userdata.get(name) or "").strip().strip('"').strip("'")
    except Exception:  # SecretNotFoundError hoặc chưa bật Notebook access
        return ""

worker_env = {name: read_secret(name) for name in REQUIRED}
missing = [name for name, value in worker_env.items() if not value]
if missing:
    raise SystemExit("Thiếu secret (hoặc chưa bật Notebook access): " + ", ".join(missing))
for name, default in OPTIONAL.items():
    worker_env[name] = read_secret(name) or default
worker_env["NESTJS_API_URL"] = worker_env["NESTJS_API_URL"].rstrip("/")
worker_env["QUEUE_NAME"] = "video-analysis"
worker_env["TMP_DIR"] = "/tmp/martial-arts-worker"
print("Đã đọc secret:", ", ".join(sorted(worker_env)))

In [ ]:
# 3. Code, thư viện, trọng số YOLO
REPO_URL = "https://github.com/Dannguyen04/MMA-TMS.git"
REPO_BRANCH = "release/final-review"  # cùng nhánh API trên Render đang chạy

import os, shutil
if not os.path.isdir("/content/MMA-TMS"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, "/content/MMA-TMS"], check=True)
else:
    subprocess.run(["git", "-C", "/content/MMA-TMS", "pull", "--ff-only"], check=True)
WORKER_DIR = "/content/MMA-TMS/python-worker"
subprocess.run(["pip", "install", "-q", "-r", f"{WORKER_DIR}/requirements.txt"], check=True)
# Dùng trọng số đã khoá trong repo, không tải bản mới từ mạng.
weights = f"{WORKER_DIR}/{worker_env['YOLO_MODEL']}.pt"
if not os.path.exists(weights) and os.path.exists(f"/content/MMA-TMS/{worker_env['YOLO_MODEL']}.pt"):
    shutil.copy(f"/content/MMA-TMS/{worker_env['YOLO_MODEL']}.pt", weights)
print("Trọng số:", weights if os.path.exists(weights) else "không có trong repo, ultralytics sẽ tự tải")
print("Commit:", subprocess.run(["git", "-C", "/content/MMA-TMS", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

In [ ]:
# 4. Kiểm tra kết nối: API (đánh thức Render, lần đầu ~1 phút) và Redis
import httpx, redis

health = httpx.get(worker_env["NESTJS_API_URL"] + "/health", timeout=120)
print("API /health:", health.status_code)

def redis_client():
    url = worker_env["REDIS_URL"]
    if url.startswith("rediss://"):
        return redis.from_url(url, decode_responses=True, ssl_cert_reqs=None)
    return redis.from_url(url, decode_responses=True)

r = redis_client()
print("Redis PING:", r.ping())
q = worker_env["QUEUE_NAME"]
waiting = r.llen(f"bull:{q}:wait") + r.llen(f"bull:{q}:waiting")
active = r.lrange(f"bull:{q}:active", 0, -1)
print(f"Job đang chờ: {waiting}; job kẹt ở active: {len(active)}")

In [ ]:
# 5. (Tuỳ chọn) Đưa job kẹt ở active về hàng chờ sau khi Colab bị ngắt giữa chừng.
# CHỈ bật khi không có worker nào khác đang chạy (ví dụ máy Dan), nếu không sẽ cướp job của worker kia.
# Video tạo hơn 1 giờ trước sẽ FAILED vì signed URL đã hết hạn: khi đó tải lại video.
REQUEUE_STUCK = False

if REQUEUE_STUCK:
    moved = 0
    while r.lmove(f"bull:{q}:active", f"bull:{q}:wait", "RIGHT", "LEFT"):
        moved += 1
    print("Đã đưa về hàng chờ:", moved, "job")
else:
    print("Bỏ qua (REQUEUE_STUCK = False)")

In [ ]:
# 6. Chạy worker. Thấy "Đang lắng nghe queue 'video-analysis'" là được. Dừng bằng nút Stop.
import re, sys

secrets = [v for k, v in worker_env.items() if k in REQUIRED and len(v) >= 6]
# Che cả mật khẩu nằm trong REDIS_URL dù worker in URL theo dạng nào.
password = re.match(r"^\w+://[^:/@]*:(.*)@", worker_env["REDIS_URL"])
if password and password.group(1):
    secrets.append(password.group(1))

def mask(line):
    for value in secrets:
        line = line.replace(value, "***")
    return line

env = {**os.environ, **worker_env, "PYTHONUNBUFFERED": "1"}
proc = subprocess.Popen([sys.executable, "worker.py"], cwd=WORKER_DIR, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(mask(line), end="")
except KeyboardInterrupt:
    print("Đang dừng worker...")
finally:
    proc.terminate()
    try:
        proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        proc.kill()
    print("Worker đã dừng, mã thoát:", proc.returncode)